In [ ]:

import os
import shutil
from pathlib import Path

from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, StringType, DoubleType, LongType, DateType
from pyspark.sql import Row
from pyspark.sql.functions import expr
from pyspark.sql.functions import *

spark = (
    SparkSession.builder
    .appName("Introduction to RDDs")
    .config("spark.master", "local[*]")
    .getOrCreate()
)


Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/03/29 16:54:18 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [2]:
sc = spark.sparkContext

In [ ]:

# 1 - parallelize an existing collection
numbers = [x for x in range(1000000)]
numbersRDD = sc.parallelize(numbers)

In [17]:


from dataclasses import dataclass

@dataclass
class StockValue:
    company: str
    date: str
    price: float

def read_stocks(filename: str):
    with open(filename, "r", encoding="utf-8") as f:
        next(f)
        return [
            StockValue(tokens[0], tokens[1], float(tokens[2]))
            for tokens in (line.strip().split(",") for line in f)
        ]

stocksRDD = sc.parallelize(read_stocks("src/main/resources/data/stocks.csv"))

In [18]:
# 2b - reading from files

stocksRDD2 = sc.textFile("src/main/resources/data/stocks.csv")

# notice to filter out header we cant drop 0 because its parallelized so we dont know which the header will be
stocksRDD2 = (
    stocksRDD2.map(lambda line : line.split(","))
    .filter(lambda tokens: tokens[0].toUpperCase() == tokens[0]) # filter out the header
    .map(lambda tokens : StockValue(tokens[0], tokens[1], float(tokens[2])))
)

In [19]:
# read from a dataframe


stocksDF = spark.read.option("header", "true").csv("src/main/resources/data/stocks.csv")

# RDD of for
stocksRDD3 = stocksDF.rdd

# or - RDD of type StockValue
stocksRDD4 = stocksDF.rdd.map(lambda row:  StockValue(row["company"], row["date"], row["price"])) 

In [20]:
# rdd to a dataframe
numbersDF = numbersRDD.map(lambda x: Row(numbers=x)).toDF()

In [27]:
# Transfomrations

msftRDD = (
    stocksRDD.filter(lambda x : x.company == "MSFT")
) # lazy transformation


msftRDD.count() # EAGER action

123

In [31]:
# distinct is also a lazy transfomration
companyNamesRDD = stocksRDD.map(lambda x: x.company).distinct()


companyNamesRDD.count()

5

In [34]:
# min and max
minMsft = msftRDD.min(key=lambda x: x.price)


print(minMsft)

StockValue(company='MSFT', date='Feb 1 2009', price=15.81)


In [35]:
# reduce
numbersRDD.reduce(lambda x, y : x + y)

499999500000

In [37]:
# grouping
# very expensive similar to Dataframes due to shuffles
stocksRDD.groupBy(lambda x: x.company)

PythonRDD[108] at RDD at PythonRDD.scala:53

In [41]:
# partitioning
# this created a new rdd !
# repartitioning is a shuffle so it is EXPENSIVE
# Best practice : partition EARLY, then process that
# Best practice : size of a partition 10-100 MB.
repartitionedStocksRDD = stocksRDD.repartition(30)

In [ ]:
# here we can write it to disk and see that this created 30 sub parquet files one for each partition
repartitionedStocksRDD.toDF().write.mode("overwrite").parquet("src/main/resources/data/rdd/stocks30")

26/03/29 17:55:48 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers
26/03/29 17:55:48 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 84.44% for 9 writers
26/03/29 17:55:48 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 76.00% for 10 writers
26/03/29 17:55:48 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 69.09% for 11 writers
26/03/29 17:55:48 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 63.33% for 12 writers
26/03/29 17:55:48 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 58.46% for 13 writers
26/03/29 17:55:48 WARN MemoryManager: Total allocation exceeds 95.

In [43]:
# coalese
# repartition rdd to less partitions than it already has
# coalese does NOT involve true shuffling
# data does not need to be moved between entire cluster
# other partitions move data to them and 15 stay in the same place
# but it usually leads to UNEVEN partition size
coalescedRDD = repartitionedStocksRDD.coalesce(15)

In [42]:
# in general coalesce when you want to reduce number of partitions with minimal cost
# and use repartition when you want to distribute EVENLY

In [44]:
# Exercises
# 1. Read the movies.json as an RDD.
# 2. show the distinct genres as an RDD
# 3. select all the movies in the Drama genre with IMDB rating > 6
# 4. show the average rating of movies by genre

In [67]:
from dataclasses import dataclass
from typing import Optional

@dataclass
class Movie:
    title: str
    rating: Optional[float]
    genre: Optional[str]

In [99]:
# exercise 1
moviesDF = spark.read.json("src/main/resources/data/movies.json")

moviesDF = (
    moviesDF.select(col("Title"), col("Major_Genre"), col("IMDB_Rating"))
    .filter(col("Major_Genre").isNotNull() & col("IMDB_Rating").isNotNull())
)

moviesRDD = moviesDF.rdd.map(lambda m: Movie(
    title=m["Title"],
    rating=float(m["IMDB_Rating"]) if m["IMDB_Rating"] is not None else None,
    genre=m["Major_Genre"]
    )
)

moviesRDD.count()


2746

In [104]:
# ex 2 
distinctMoviesRDD = moviesRDD.map(lambda movie: Row(genre=movie.genre)).distinct()

distinctMoviesRDD.toDF().show()

+-------------------+
|              genre|
+-------------------+
|              Drama|
|             Comedy|
|            Musical|
|  Thriller/Suspense|
|             Action|
|    Romantic Comedy|
|          Adventure|
|            Western|
|             Horror|
|        Documentary|
|       Black Comedy|
|Concert/Performance|
+-------------------+



In [105]:
# ex 3
# IMDB rating > 6
filteredMoviesRDD = moviesRDD.filter(lambda movie : movie.rating >  6) 

filteredMoviesRDD.toDF().show(5)

+-------+------+--------------------+
|  genre|rating|               title|
+-------+------+--------------------+
|  Drama|   6.9|First Love, Last ...|
| Comedy|   6.8|I Married a Stran...|
|Musical|   7.5|             Oliver!|
|  Drama|   8.9|        12 Angry Men|
|  Drama|   8.1|      Twelve Monkeys|
+-------+------+--------------------+
only showing top 5 rows



In [ ]:
from statistics import mean

groupedMoviesRDD = moviesRDD.filter(
    lambda x: x.genre is not None
).groupBy(
    lambda x: x.genre
)

avgByGenreRDD = groupedMoviesRDD.mapValues(
    lambda movies: mean([movie.rating for movie in movies if movie.rating is not None])
)

avgByGenreRDD.toDF(["genre", "avg_rating"]).show(5)

+-----------------+------------------+
|            genre|        avg_rating|
+-----------------+------------------+
|            Drama| 6.773441734417344|
|           Comedy| 5.853858267716536|
|          Musical|             6.448|
|Thriller/Suspense|6.3609442060085835|
|        Adventure| 6.345019920318725|
+-----------------+------------------+
only showing top 5 rows



In [110]:
groupedMoviesRDD2 = moviesRDD.groupBy(lambda movie: movie.genre)

avgRatingByGenreRDD = groupedMoviesRDD2.map(lambda genre, movies : Row(genre=genre, avgRating=mean(movies.rating)))

avgByGenreRDD.toDF().show()

+-------------------+------------------+
|                 _1|                _2|
+-------------------+------------------+
|              Drama| 6.773441734417344|
|             Comedy| 5.853858267716536|
|            Musical|             6.448|
|  Thriller/Suspense|6.3609442060085835|
|          Adventure| 6.345019920318725|
|             Action| 6.114795918367347|
|    Romantic Comedy|5.8730769230769235|
|             Horror| 5.676076555023924|
|            Western|6.8428571428571425|
|        Documentary|6.9972972972972975|
|       Black Comedy|           6.81875|
|Concert/Performance|             6.325|
+-------------------+------------------+

